In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
import os
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, Concatenate, Input
from tensorflow.keras.callbacks import EarlyStopping

# --- Set random seeds for reproducibility ---
os.environ['PYTHONHASHSEED'] = '0'
np.random.seed(42)
tf.random.set_seed(42)

# Helper function to downcast numeric data types to save memory 
def downcast_dataframe(df):
    """Downcasts numeric data types to save memory."""
    print("Downcasting numeric columns to save memory...")
    for col in df.select_dtypes(include=np.number).columns:
        df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
    return df

print("--- Starting Sequential Fusion Model Training (Corrected) ---")

# --- Load and Merge Data ---
print("Loading and merging data...")
try:
    X_transaction_df = pd.read_csv("../data/processed/train_transaction_sample.csv")
    X_identity_df = pd.read_csv("../data/processed/train_identity_sample.csv")
    y = X_transaction_df['isFraud'].values
except FileNotFoundError as e:
    print(f"Error: {e}. Please ensure data files are in the correct directory.")
    exit()

X_df = X_transaction_df.merge(X_identity_df, on="TransactionID", how="left")
print(f"Shape of the merged dataframe: {X_df.shape}")
transaction_ids = X_df['TransactionID'].values

llm_embeddings_path = "../data/processed/llm_embeddings.csv"
gnn_path = "../data/processed/gnn_embeddings.csv"

# Load and merge LLM embeddings
if os.path.exists(llm_embeddings_path):
    llm_embeddings_full = pd.read_csv(llm_embeddings_path, low_memory=False)
    llm_embeddings_full = downcast_dataframe(llm_embeddings_full)
    llm_embeddings = llm_embeddings_full[llm_embeddings_full['TransactionID'].isin(transaction_ids)]
    X_df = X_df.merge(llm_embeddings, on="TransactionID", how='left')
    print(f"LLM embeddings loaded and merged. Shape: {X_df.shape}")
else:
    print(f"Warning: {llm_embeddings_path} not found. Skipping LLM embeddings.")

# Load and merge GNN embeddings
if os.path.exists(gnn_path):
    gnn_full = pd.read_csv(gnn_path, low_memory=False)
    gnn_full = downcast_dataframe(gnn_full)
    gnn = gnn_full[gnn_full['TransactionID'].isin(transaction_ids)]
    X_df = X_df.merge(gnn, on="TransactionID", how='left')
    print(f"GNN embeddings loaded and merged. Shape: {X_df.shape}")
else:
    print(f"Warning: {gnn_path} not found. Skipping GNN embeddings.")

# --- Prepare Features & Labels ---
X_df = X_df.drop(columns=['TransactionID', 'isFraud'], errors="ignore")
leaky_cols = ['DeviceInfo', 'card1', 'id_31', 'id_33', 'Prompt']
X_df = X_df.drop(columns=leaky_cols, errors='ignore')

HIGH_CARDINALITY_THRESHOLD = 50
categorical_cols_all = X_df.select_dtypes(include=['object', 'category']).columns.tolist()
high_cardinality_cols = [col for col in categorical_cols_all if X_df[col].nunique() > HIGH_CARDINALITY_THRESHOLD]
categorical_cols = [col for col in categorical_cols_all if col not in high_cardinality_cols]
X_df = X_df.drop(columns=high_cardinality_cols, errors='ignore')

gnn_cols = [col for col in X_df.columns if col.startswith('gnn_embed_')]
llm_embed_cols = [col for col in X_df.columns if col.startswith('LLM_embed_')]
tabular_numeric_cols = [col for col in X_df.select_dtypes(include=np.number).columns.tolist() if col not in gnn_cols and col not in llm_embed_cols]

for col in tabular_numeric_cols + gnn_cols + llm_embed_cols:
    X_df[col] = pd.to_numeric(X_df[col], errors='coerce').fillna(0)

# --- Preprocessing Pipeline ---
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, tabular_numeric_cols),
        ('cat', categorical_transformer, categorical_cols)],
    remainder='drop',
)

# --- Split the data and apply preprocessor ---
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X_df, y, test_size=0.2, stratify=y, random_state=42
)

# Tabular data is processed
X_train_processed_tabular = preprocessor.fit_transform(X_train_df)
X_test_processed_tabular = preprocessor.transform(X_test_df)

# GNN and LLM embeddings are extracted directly
X_train_gnn = X_train_df[gnn_cols].values
X_test_gnn = X_test_df[gnn_cols].values
X_train_llm = X_train_df[llm_embed_cols].values
X_test_llm = X_test_df[llm_embed_cols].values

print("\nFitting and transforming data...")
print(f"Shape of X_train processed tabular data: {X_train_processed_tabular.shape}")
print(f"Shape of X_train GNN embeddings: {X_train_gnn.shape}")
print(f"Shape of X_train LLM embeddings: {X_train_llm.shape}")

num_non_fraud = np.sum(y_train == 0)
num_fraud = np.sum(y_train == 1)
# class weight calculation for better balance
total = num_non_fraud + num_fraud
weight_for_0 = (1 / num_non_fraud) * (total / 2.0)
weight_for_1 = (1 / num_fraud) * (total / 2.0)
class_weight = {0: weight_for_0, 1: weight_for_1}

print(f"Calculated class weights: {class_weight}")

# Define a separate input for each data type
input_tabular = Input(shape=(X_train_processed_tabular.shape[1],), name='tabular_input')
input_gnn = Input(shape=(X_train_gnn.shape[1],), name='gnn_input')
input_llm = Input(shape=(X_train_llm.shape[1],), name='llm_input')

# Concatenate all inputs into a single layer
concatenated_features = Concatenate(name='final_concat')([input_tabular, input_gnn, input_llm])

# Build the main model
x = Dense(256, activation='relu')(concatenated_features)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=[input_tabular, input_gnn, input_llm], outputs=output)

model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=[
                  'accuracy',
                  tf.keras.metrics.AUC(name='auc'),
                  tf.keras.metrics.Precision(name='precision'),
                  tf.keras.metrics.Recall(name='recall')
              ])

model.summary()

early_stopping = EarlyStopping(monitor='val_auc', patience=10, mode='max', restore_best_weights=True)

print("\nStarting Sequential Fusion model training...")
# Pass the data as a dictionary to match the named inputs
history = model.fit(
    {'tabular_input': X_train_processed_tabular, 'gnn_input': X_train_gnn, 'llm_input': X_train_llm},
    y_train,
    epochs=100,
    batch_size=32,
    validation_data=({'tabular_input': X_test_processed_tabular, 'gnn_input': X_test_gnn, 'llm_input': X_test_llm}, y_test),
    class_weight=class_weight,
    callbacks=[early_stopping],
    verbose=1
)

print("Sequential Fusion model training complete.")

# --- Evaluation ---
print("\n--- Model Evaluation ---")
probs = model.predict({'tabular_input': X_test_processed_tabular, 'gnn_input': X_test_gnn, 'llm_input': X_test_llm}).flatten()
y_pred = (probs >= 0.5).astype(int)

# Removed `zero_division` from accuracy_score as it is not a supported parameter
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
conf_matrix = confusion_matrix(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probs)

print(f"Accuracy: {accuracy:.3f}")
print(f"Precision: {precision:.3f}")
print(f"Recall: {recall:.3f}")
print(f"F1 Score: {f1:.3f}")
print("Confusion Matrix:")
print(conf_matrix)
print(f"ROC AUC Score: {roc_auc:.3f}")

--- Starting Sequential Fusion Model Training (Corrected) ---
Loading and merging data...
Shape of the merged dataframe: (50000, 434)
Downcasting numeric columns to save memory...


C:\Users\aishu\AppData\Local\Temp\ipykernel_22244\4215316974.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')


LLM embeddings loaded and merged. Shape: (50000, 1971)
Downcasting numeric columns to save memory...


C:\Users\aishu\AppData\Local\Temp\ipykernel_22244\4215316974.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_22244\4215316974.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_22244\4215316974.py:25: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df[col] = pd.to_numeric(df[col], downcast='float', errors='ignore')
C:\Users\aishu\AppData\Local\Temp\ipykernel_22244\4215316974.py:25: FutureWarning: errors='ignore' is d

GNN embeddings loaded and merged. Shape: (50000, 2035)

Fitting and transforming data...
Shape of X_train processed tabular data: (40000, 1996)
Shape of X_train GNN embeddings: (40000, 64)
Shape of X_train LLM embeddings: (40000, 1536)
Calculated class weights: {0: 0.5185915054711404, 1: 13.94700139470014}


...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ tabular_input       │ (None, 1996)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gnn_input           │ (None, 64)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ llm_input           │ (None, 1536)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ final_concat        │ (None, 3596)      │          0 │ tabular_input[0]… │
│ (Concatenate)       │                   │            │ gnn_input[0][0],  │
│                     │                   │            │ llm_input[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 256)       │    920,832 │ final_concat[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 256)       │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 128)       │     32,896 │ dropout_2[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_3 (Dropout) │ (None, 128)       │          0 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 64)        │      8,256 │ dropout_3[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         65 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 962,049 (3.67 MB)

 Trainable params: 962,049 (3.67 MB)

 Non-trainable params: 0 (0.00 B)


Starting Sequential Fusion model training...
Epoch 1/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 10s 6ms/step - accuracy: 0.6507 - auc: 0.6581 - loss: 0.6828 - precision: 0.0594 - recall: 0.5893 - val_accuracy: 0.8952 - val_auc: 0.7747 - val_loss: 0.5222 - val_precision: 0.1586 - val_recall: 0.4457
Epoch 2/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.7563 - auc: 0.7498 - loss: 0.5875 - precision: 0.0853 - recall: 0.5962 - val_accuracy: 0.7868 - val_auc: 0.8209 - val_loss: 0.5302 - val_precision: 0.1103 - val_recall: 0.6992
Epoch 3/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.7639 - auc: 0.7640 - loss: 0.5647 - precision: 0.0908 - recall: 0.6192 - val_accuracy: 0.8720 - val_auc: 0.8145 - val_loss: 0.4769 - val_precision: 0.1556 - val_recall: 0.5794
Epoch 4/100
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 7s 6ms/step - accuracy: 0.7610 - auc: 0.7798 - loss: 0.5565 - precision: 0.0936 - recall: 0.6527 - val_accuracy: 0.8169 - val_auc: 0.8175 - val_loss: 0.4698 - val_precision: 0